In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

possible_paths = [
    Path("data/processed/ChargingRecords_cleaned.csv"),
    Path("ChargingRecords_cleaned.csv"),
    Path("../data/processed/ChargingRecords_cleaned.csv")
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "ChargingRecords_cleaned.csv not found. "
        "Check your data/processed folder."
    )

df = pd.read_csv(DATA_PATH)

for col in ["StartDatetime", "EndDatetime"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Shape: (72799, 13)


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,2022-09-15 23:59:13,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,2022-09-14 21:31:04,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:30,2022-09-14 19:54:29,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:51,2022-09-30 00:16:42,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:15,2022-09-26 00:30:14,300,33.81


In [9]:
df["StartHour"] = df["StartDatetime"].dt.hour
df["StartDayOfWeek"] = df["StartDatetime"].dt.dayofweek
df["StartMonth"] = df["StartDatetime"].dt.month
df["StartDay"] = df["StartDatetime"].dt.day

df["IsWeekend"] = (
    df["StartDayOfWeek"] >= 5
).astype("int8")

df["StartDayName"] = df["StartDatetime"].dt.day_name()

print("Time features created!")
display(
    df[
        [
            "StartDatetime",
            "StartHour",
            "StartDayOfWeek",
            "StartMonth",
            "IsWeekend",
            "StartDayName"
        ]
    ].head()
)

Time features created!


,StartDatetime,StartHour,StartDayOfWeek,StartMonth,IsWeekend,StartDayName
0,2022-09-15 20:54:02,20,3,9,0,Thursday
1,2022-09-14 20:01:05,20,2,9,0,Wednesday
2,2022-09-14 18:54:30,18,2,9,0,Wednesday
3,2022-09-29 18:32:51,18,3,9,0,Thursday
4,2022-09-25 19:30:15,19,6,9,1,Sunday


In [10]:
df["Duration"] = pd.to_numeric(
    df["Duration"], errors="coerce"
)

df["DurationHours"] = df["Duration"] / 60

print("Duration features created!")
display(df[["Duration", "DurationHours"]].head())

Duration features created!


,Duration,DurationHours
0,185,3.083333
1,90,1.500000
2,60,1.000000
3,344,5.733333
4,300,5.000000


In [11]:
df["Demand"] = pd.to_numeric(
    df["Demand"], errors="coerce"
)

demand_threshold = df["Demand"].median()

df["DemandCategory"] = np.where(
    df["Demand"] >= demand_threshold,
    "High",
    "Normal"
)

print("Demand threshold:", demand_threshold)
print("\nDemand category counts:")
display(df["DemandCategory"].value_counts())

Demand threshold: 14.1

Demand category counts:


DemandCategory
High      36434
Normal    36365
Name: count, dtype: int64

In [12]:
categorical_columns = [
    col for col in [
        "ChargerCompany",
        "Location",
        "ChargerType"
    ]
    if col in df.columns
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("Categorical encoding completed!")
print("New shape:", df.shape)
display(df.head())

Categorical encoding completed!
New shape: (72799, 32)


,UserID,ChargerID,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,...,Location_golf,Location_hotel,Location_market,Location_public area,Location_public institution,Location_public parking lot,Location_resort,Location_restaurant,Location_sightseeing,ChargerType_1
0,0,1,15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,2022-09-15 23:59:13,185,20.36,...,0,1,0,0,0,0,0,0,0,0
1,0,1,14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,2022-09-14 21:31:04,90,10.19,...,0,1,0,0,0,0,0,0,0,0
2,0,1,14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:30,2022-09-14 19:54:29,60,6.78,...,0,1,0,0,0,0,0,0,0,0
3,0,1,29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:51,2022-09-30 00:16:42,344,37.65,...,0,1,0,0,0,0,0,0,0,0
4,0,1,25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:15,2022-09-26 00:30:14,300,33.81,...,0,1,0,0,0,0,0,0,0,0


In [13]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
display(
    df.isnull().sum()
      .to_frame("Missing_Count")
      .query("Missing_Count > 0")
)

print("\nFeature columns:")
print(df.columns.tolist())

print("\nDemand category distribution:")
display(df["DemandCategory"].value_counts())

Dataset shape: (72799, 32)

Missing values:


,Missing_Count



Feature columns:
['UserID', 'ChargerID', 'StartDay', 'StartTime', 'EndDay', 'EndTime', 'StartDatetime', 'EndDatetime', 'Duration', 'Demand', 'StartHour', 'StartDayOfWeek', 'StartMonth', 'IsWeekend', 'StartDayName', 'DurationHours', 'DemandCategory', 'ChargerCompany_1', 'Location_apartment', 'Location_bus garage', 'Location_camping', 'Location_company', 'Location_golf', 'Location_hotel', 'Location_market', 'Location_public area', 'Location_public institution', 'Location_public parking lot', 'Location_resort', 'Location_restaurant', 'Location_sightseeing', 'ChargerType_1']

Demand category distribution:


DemandCategory
High      36434
Normal    36365
Name: count, dtype: int64

In [14]:
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "ChargingRecords_features.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("Feature engineering completed successfully!")
print("Saved to:", OUTPUT_PATH.resolve())
print("Final dataset shape:", df.shape)

Feature engineering completed successfully!
Saved to: C:\Users\kavee\OneDrive\Desktop\EV Charging Demand Prediction and Usage Pattern Analysis\notebook\data\processed\ChargingRecords_features.csv
Final dataset shape: (72799, 32)
